# Tahap 7 - Tokenisasi dan Fine-Tuning IndoBERT

Notebook ini menjalankan tahap training IndoBERT untuk analisis sentimen JakOne Mobile. Tahap ini hanya memakai split train dan validation untuk fine-tuning serta menyiapkan test set tanpa evaluasi final.

## 1. Import Library dan Script Training

Cell ini memuat library utama dan mengimpor fungsi dari `src/07_finetune_indobert.py` agar alur notebook sama dengan script Python.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPT_PATH = PROJECT_ROOT / "src" / "07_finetune_indobert.py"

spec = importlib.util.spec_from_file_location("finetune_indobert", SCRIPT_PATH)
finetune = importlib.util.module_from_spec(spec)
spec.loader.exec_module(finetune)

## 2. Load Dataset dan Encode Label

Dataset final dibaca dari `data/final/jakone_modeling_master.csv`. Kolom yang dipakai adalah `clean_review`, `label`, dan `split_set`. Label dikodekan menjadi `negatif=0`, `netral=1`, dan `positif=2`.

In [ ]:
finetune.set_seed(finetune.SEED)
finetune.prepare_output_dirs()
finetune.validate_input_file()

df = finetune.load_dataset()
train_df, val_df, test_df = finetune.split_dataset(df)
finetune.save_label_mapping()

print("Jumlah train:", len(train_df))
print("Jumlah validation:", len(val_df))
print("Jumlah test disiapkan:", len(test_df))
df[[finetune.TEXT_COLUMN, finetune.LABEL_COLUMN, finetune.SPLIT_COLUMN, "labels"]].head()

## 3. Tokenisasi IndoBERT

Tokenizer `indobenchmark/indobert-base-p2` digunakan pada kolom `clean_review` dengan `max_length=128`, `padding="max_length"`, dan `truncation=True`.

In [ ]:
tokenizer = finetune.BertTokenizer.from_pretrained(finetune.MODEL_NAME)
train_loader, val_loader, test_loader = finetune.build_dataloaders(tokenizer, train_df, val_df, test_df)

print("Batch train:", len(train_loader))
print("Batch validation:", len(val_loader))
print("Batch test disiapkan tanpa evaluasi:", len(test_loader))

## 4. Class Weight

Class weight dihitung dari data train untuk menangani ketidakseimbangan label, terutama kelas `netral` yang proporsinya lebih kecil.

In [ ]:
device = finetune.torch.device("cuda" if finetune.torch.cuda.is_available() else "cpu")
class_weights_tensor, class_weights = finetune.calculate_class_weights(train_df, device)

pd.DataFrame({
    "label_id": [0, 1, 2],
    "label": ["negatif", "netral", "positif"],
    "class_weight": class_weights,
})

## 5. Fine-Tuning IndoBERT

Model `BertForSequenceClassification` dilatih maksimal 5 epoch menggunakan AdamW, scheduler linear warmup 10%, weighted cross entropy, dan early stopping berdasarkan validation F1 macro dengan patience 2.

In [ ]:
model = finetune.BertForSequenceClassification.from_pretrained(
    finetune.MODEL_NAME,
    num_labels=3,
    id2label=finetune.ID2LABEL,
    label2id=finetune.LABEL2ID,
)
model.to(device)

loss_fn = finetune.torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = finetune.torch.optim.AdamW(
    model.parameters(),
    lr=finetune.LEARNING_RATE,
    weight_decay=finetune.WEIGHT_DECAY,
)

total_training_steps = len(train_loader) * finetune.EPOCHS
warmup_steps = int(0.1 * total_training_steps)
scheduler = finetune.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

## 6. Training Loop Manual

Cell ini menjalankan loop training manual PyTorch. Setiap epoch mencatat train loss, validation loss, validation accuracy, dan validation F1 macro. Model terbaik disimpan berdasarkan validation F1 macro tertinggi.

In [ ]:
best_f1 = -1.0
best_epoch = 0
patience_counter = 0
log_rows = []

for epoch in range(1, finetune.EPOCHS + 1):
    train_loss = finetune.train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_accuracy, val_f1_macro = finetune.evaluate_validation(model, val_loader, loss_fn, device)

    log_rows.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "val_loss": round(val_loss, 6),
        "val_accuracy": round(val_accuracy, 6),
        "val_f1_macro": round(val_f1_macro, 6),
    })

    print(log_rows[-1])

    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        best_epoch = epoch
        patience_counter = 0
        model.save_pretrained(finetune.MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(finetune.MODEL_OUTPUT_DIR)
        finetune.save_label_mapping()
    else:
        patience_counter += 1
        if patience_counter >= finetune.PATIENCE:
            print(f"Early stopping pada epoch {epoch}")
            break

training_log = finetune.save_training_log(log_rows)
print("Epoch terbaik:", best_epoch)
print("Validation F1 macro terbaik:", round(best_f1, 6))
training_log

## 7. Grafik Train Loss vs Validation Loss

Grafik ini membantu melihat perubahan loss train dan validation selama proses fine-tuning.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_log["epoch"], training_log["train_loss"], marker="o", label="Train Loss")
plt.plot(training_log["epoch"], training_log["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train Loss vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Grafik Validation F1 Macro

Grafik ini menampilkan validation F1 macro per epoch. Nilai ini digunakan sebagai dasar penyimpanan model terbaik dan early stopping.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_log["epoch"], training_log["val_f1_macro"], marker="o", color="green")
plt.xlabel("Epoch")
plt.ylabel("Validation F1 Macro")
plt.title("Validation F1 Macro per Epoch")
plt.grid(True, alpha=0.3)
plt.show()

## 9. Output Tahap 7

Output tahap ini adalah model IndoBERT terbaik di `outputs/model_indobert_jakone/` dan log training di `outputs/evaluation/training_log.csv`. Evaluasi final test set tidak dilakukan pada notebook ini.